# The question catalogue


**What this is (instruction 73, phase 3).** Every column of the Toxoplasma table that can be a label, turned into one candidate question by its information category, with the admission tests recorded. A list to prune, not results: no question is scored here.

**Admission tests recorded per row.** *Answerable*: at least 150 labelled genes (the harness default for a frozen test). *Non-redundant*: the first column of its measured redundancy group (every pair in a group has |Spearman| >= 0.8); the others are listed as restating it. *Registered*: already in the question registry (scored through the harness).

Built 2026-11-08.

In [1]:
import os, re, numpy as np, pandas as pd
from parascape import information_categories as IC, question_registry as Q, redundancy as R
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet'))
groups = R.groups('Tg').set_index('column')
registered = {}
for q in Q.QUESTIONS.values():
    col = str(q.label_source).rsplit(':', 1)[-1].split(' ')[0]
    registered.setdefault(col, []).append(q.id)
print(len(nodes.columns), 'columns;', len(Q.QUESTIONS), 'registered questions')

443 columns; 21 registered questions


## 1 · One candidate question per label column

Wording by category. Identity, sequence, study-effort and pure annotation-bookkeeping columns are not questions about biology and are skipped.

In [2]:
WORDING = {
  'fitness': 'Is the gene needed for growth under {x}?',
  'phenotype': 'What happens when the gene is perturbed ({x})?',
  'place': 'Where is the protein ({x})?',
  'function': 'What does the gene do ({x})?',
  'transcript': 'Does the gene respond in {x}?',
  'translation': 'Is the gene translated differently in {x}?',
  'protein_abundance': 'How much protein is there in {x}?',
  'modification': 'Is the protein modified ({x})?',
  'interaction': 'What does the protein bind ({x})?',
  'host_response': 'What does the host do in response ({x})?',
}
SKIP = {'identity', 'sequence', 'study_effort', 'conservation', 'structure', 'domain', 'annotation'}
rows = []
for c in nodes.columns:
    try:
        cats = tuple(IC.categories_of(c))
    except Exception:
        cats = ()
    cat = next((k for k in cats if k in WORDING), None)
    if cat is None or set(cats) & SKIP and cat not in WORDING:
        continue
    s = nodes[c]
    labelled = int(s.notna().sum())
    numeric = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
    kind = 'value' if numeric and s.nunique() > 12 else 'class'
    rows.append(dict(column=c, category=cat, kind=kind, labelled=labelled,
                     classes=int(s.nunique()) if kind == 'class' else None,
                     question=WORDING[cat].format(x=c.replace('_', ' ')),
                     registered=', '.join(registered.get(c, [])),
                     group=int(groups.loc[c, 'group']) if c in groups.index else None))
cat_df = pd.DataFrame(rows)
print(len(cat_df), 'candidate questions')
cat_df.category.value_counts().to_frame('questions')

384 candidate questions


,questions
category,
transcript,175
translation,46
fitness,43
host_response,26
protein_abundance,26
modification,17
interaction,17
place,16
phenotype,16


## 2 · Admission: answerable and non-redundant

Within a measured group, the column with the most labelled genes is kept as the question; the others are marked as restating it. Columns outside any measured group (categorical, or not measured on enough genes to correlate) stand on their own.

In [3]:
cat_df['answerable'] = cat_df.labelled >= MIN_LABELLED
cat_df = cat_df.sort_values(['group', 'labelled'], ascending=[True, False])
first = cat_df.dropna(subset=['group']).groupby('group').column.first()
cat_df['restates'] = [first.get(g) if pd.notna(g) and first.get(g) != c else '' for g, c in zip(cat_df.group, cat_df.column)]
cat_df['non_redundant'] = cat_df.restates == ''
cat_df['admitted'] = cat_df.answerable & cat_df.non_redundant
admitted = cat_df[cat_df.admitted]
independent = admitted.group.nunique() + int(admitted.group.isna().sum())
summary = dict(candidates=len(cat_df), answerable=int(cat_df.answerable.sum()),
               restating_another=int((~cat_df.non_redundant).sum()), admitted=len(admitted),
               effective_independent_questions=int(independent),
               already_registered=int((cat_df.registered != '').sum()))
cat_df.sort_values(['category', 'labelled'], ascending=[True, False]).to_csv(os.path.join(OUT, 'catalogue.tsv'), sep='\t', index=False)
summary

{'candidates': 384, 'answerable': 342, 'restating_another': 165, 'admitted': 177, 'effective_independent_questions': 177, 'already_registered': 2}

In [4]:
admitted.groupby('category').agg(questions=('column', 'size'), median_labelled=('labelled', 'median'),
                                 registered=('registered', lambda s: int((s != '').sum())))

,questions,median_labelled,registered
category,,,
fitness,30,7393,0
function,2,4726,0
host_response,3,252,0
interaction,15,1306,0
modification,13,660,0
phenotype,6,318,1
place,16,3827,0
protein_abundance,8,3005,0
transcript,62,6504,0


## 3 · The gaps that must be declared

Questions Einar asked (instruction 73) that no column in the table can label: declared, not fudged.

In [5]:
gaps = pd.DataFrame([
  dict(question='favourable immune response in humans', why='no human screen exists'),
  dict(question='involvement in autophagy', why='no genome-wide autophagy assay in the table'),
  dict(question='protein scavenging from the host', why='no scavenging screen in the table'),
  dict(question='sporozoite-to-tachyzoite switching', why='no stage-switch screen; sporozoite RNA in one study only')])
gaps.to_csv(os.path.join(OUT, 'declared_gaps.tsv'), sep='\t', index=False)
gaps

,question,why
0,favourable immune response in humans,no human screen exists
1,involvement in autophagy,no genome-wide autophagy assay in the table
2,protein scavenging from the host,no scavenging screen in the table
3,sporozoite-to-tachyzoite switching,no stage-switch screen; sporozoite RNA in one study only


In [6]:
import json
json.dump(summary, open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
admitted.sort_values('labelled', ascending=False).head(40)[['question', 'category', 'labelled', 'registered']]

,question,category,labelled,registered
310,Does the gene respond in antisense expression percentile?,transcript,8140,
345,What does the protein bind (n ipms partners)?,interaction,8140,
34,Where is the protein (compartment)?,place,8140,
344,What does the protein bind (n xlink partners)?,interaction,8140,
0,Where is the protein (lopit confident)?,place,8140,
320,What does the gene do (has ec)?,function,8140,
41,Where is the protein (compartment best)?,place,8140,
40,Where is the protein (compartment source)?,place,8140,
39,Where is the protein (ortholopit accepted)?,place,8140,
20,Is the protein modified (has phospho)?,modification,8140,
